# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library, referencing all data entities by their `@id` fields as per the Croissant specification.

### Dataset Source
The dataset is defined by a Croissant schema accessible at the following URL:

https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review the available record sets and their structure using their `@id` fields.

The dataset may contain multiple record sets. We'll enumerate them and display their fields (columns) by referencing their `@id`.

In [ ]:
# List all record sets (@ids)
record_set_ids = [rs['@id'] for rs in metadata.record_sets]
print('Record sets in the dataset:')
for rs in metadata.record_sets:
    print(f"@id: {rs['@id']}, name: {rs.get('name', 'N/A')}")
    # List fields for this record set
    if 'fields' in rs:
        print("  Fields:")
        for f in rs['fields']:
            print(f"    - @id: {f['@id']}, name: {f.get('name', 'N/A')}, dataType: {f.get('dataType', 'N/A')}")
    print()

## 3. Data Extraction
Extract records from one or more record sets into pandas DataFrames, using the record set and field `@id`s identified in the overview.

In [ ]:
# Prepare to extract data for each record set by @id
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records from record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records. Columns (field @ids):")
        print(df.columns.tolist())
        print(df.head(3))
    else:
        print("No records found or record set not tabular.")
    print()

# For subsequent operations, select the primary (first) record set
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"Using record set @id for further exploration: {main_record_set_id}")
else:
    main_record_set_id = None
    print("No record sets were found.")

## 4. Exploratory Data Analysis (EDA)
Explore and process the dataset.

- We'll filter numeric records, normalize a numeric field, and group by a categorical field, referencing fields by their `@id`.

In [ ]:
# Identify a numeric field and a categorical field from the record set fields
numeric_field_id = None
group_field_id = None
if main_record_set_id and main_record_set_id in dataframes:
    main_df = dataframes[main_record_set_id]
    # Search the metadata for appropriate fields
    # We'll choose numeric and group-able fields from the metadata definition
    for rs in metadata.record_sets:
        if rs['@id'] == main_record_set_id:
            for f in rs.get('fields', []):
                # Look for the first Integer/Float field
                dt = str(f.get('dataType', '')).lower()
                if not numeric_field_id and (dt.startswith('schema:integer') or dt.startswith('schema:float') or dt == 'integer' or dt=='float'):
                    numeric_field_id = f['@id']
                # Look for first categorical/string/text field for grouping
                if not group_field_id and (dt.startswith('schema:text') or dt.startswith('schema:string') or dt == 'text' or dt=='string'):
                    group_field_id = f['@id']
                if numeric_field_id and group_field_id:
                    break
        if numeric_field_id and group_field_id:
            break
    print(f"Numeric field @id: {numeric_field_id}")
    print(f"Group (categorical) field @id: {group_field_id}")
    
    # Proceed if we found suitable fields
    if numeric_field_id and numeric_field_id in main_df.columns:
        # Remove obvious missing/non-numeric values (if any)
        filtered_df = main_df[pd.to_numeric(main_df[numeric_field_id], errors='coerce').notnull()]
        filtered_df[numeric_field_id] = pd.to_numeric(filtered_df[numeric_field_id])
        # Set a threshold to filter
        threshold = filtered_df[numeric_field_id].mean()
        filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
        print(f"Filtered records in '{numeric_field_id}' > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # If group field is available, group and aggregate
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped average of '{numeric_field_id}' by '{group_field_id}':")
            print(grouped_df)
    else:
        print("Could not find suitable numeric field for EDA.")
else:
    print("Main record set DataFrame unavailable or record set id not set.")

## 5. Visualization
Visualize numeric field distribution and grouped means, referencing all fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Show histogram for the main numeric field
if main_record_set_id and main_record_set_id in dataframes and numeric_field_id and numeric_field_id in dataframes[main_record_set_id].columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(dataframes[main_record_set_id][numeric_field_id].dropna().astype(float), bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If group field exists, plot group-wise means
    if group_field_id and group_field_id in dataframes[main_record_set_id].columns:
        grouped_plot = dataframes[main_record_set_id].groupby(group_field_id)[numeric_field_id].mean().reset_index()
        plt.figure(figsize=(8, 4))
        sns.barplot(data=grouped_plot, x=group_field_id, y=numeric_field_id)
        plt.title(f"Average {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("Unable to plot: required numeric/group fields not found.")

## 6. Conclusion

- We loaded the FAIR^2 colorectal cancer survivors dataset and explored record sets, fields, and extracted the main data table via `mlcroissant` using entity `@id`s.
- Numeric and categorical field analyses were performed along with EDA and basic visualization.
- This workflow forms a reproducible and FAIR data science pipeline leveraging the Croissant standard.